# Class–axiom generator
For each ontology: every non-deprecated named class, its IRI, `rdfs:label`, and the axioms asserted on it (`rdfs:subClassOf` / `owl:equivalentClass`). SPARQL retrieves the raw axiom parts; Python renders them as readable text.

In [1]:
from pathlib import Path
import pandas as pd
from rdflib import Graph, BNode, URIRef
from rdflib.namespace import OWL, RDFS

SRC = Path.cwd().resolve().parent / "src"
DATA = SRC / "data"
TTL = {"bfo": "bfo-core.ttl", "ies": "ies.ttl", "ccom": "ccom.ttl",
       "qudt": "qudt.ttl", "ccot": "ccot.ttl", "to": "time.ttl"}

In [2]:
QUERY = """
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?iri ?label ?rel ?target ?prop ?kind ?filler
WHERE {
  ?iri a ?classType .
  VALUES ?classType { owl:Class rdfs:Class }            # IES declares rdfs:Class
  FILTER(isIRI(?iri))                                   # named classes only
  FILTER NOT EXISTS {                                   # drop deprecated classes
    ?iri owl:deprecated ?d . FILTER(STR(?d) = "true")
  }
  OPTIONAL { ?iri rdfs:label ?label }
  OPTIONAL {                                            # axioms on the class
    ?iri ?rel ?target .
    VALUES ?rel { rdfs:subClassOf owl:equivalentClass }
    OPTIONAL {                                          # restriction parts
      ?target a owl:Restriction ; owl:onProperty ?prop .
      OPTIONAL {
        ?target ?kind ?filler .
        VALUES ?kind { owl:someValuesFrom owl:allValuesFrom owl:hasValue
                       owl:onClass owl:cardinality owl:qualifiedCardinality
                       owl:minCardinality owl:maxCardinality
                       owl:minQualifiedCardinality owl:maxQualifiedCardinality }
      }
    }
  }
}
"""

In [3]:
KIND = {OWL.someValuesFrom: "some", OWL.allValuesFrom: "only", OWL.hasValue: "value",
        OWL.onClass: "onClass", OWL.cardinality: "exactly", OWL.qualifiedCardinality: "exactly",
        OWL.minCardinality: "min", OWL.maxCardinality: "max",
        OWL.minQualifiedCardinality: "min", OWL.maxQualifiedCardinality: "max"}

def show(x):
    return "[complex class]" if isinstance(x, BNode) else str(x)

def render(r):
    if r.rel is None:
        return None
    head = "SubClassOf " if r.rel == RDFS.subClassOf else "EquivalentTo "
    if isinstance(r.target, URIRef):
        return head + str(r.target)
    if r.prop is not None:
        tail = f"{KIND[r.kind]} {show(r.filler)}" if r.kind is not None else "[restriction]"
        return head + f"{r.prop} {tail}"
    return head + "[complex class expression]"

for abbr, fname in TTL.items():
    g = Graph().parse(SRC / fname)
    classes = {}
    for r in g.query(QUERY):
        c = classes.setdefault(str(r.iri), {"label": "", "axioms": set()})
        if r.label is not None and not c["label"]:
            c["label"] = str(r.label)
        ax = render(r)
        if ax:
            c["axioms"].add(ax)
    df = pd.DataFrame([{"iri": iri, "label": v["label"], "axiom": " | ".join(sorted(v["axioms"]))}
                       for iri, v in sorted(classes.items())])
    df.to_excel(DATA / f"{abbr}-axioms.xlsx", index=False)
    print(f"{abbr:>4}: {len(df):4d} classes -> {abbr}-axioms.xlsx")

 bfo:   36 classes -> bfo-axioms.xlsx


 ies:  510 classes -> ies-axioms.xlsx
ccom:   28 classes -> ccom-axioms.xlsx
qudt:  175 classes -> qudt-axioms.xlsx
ccot:   33 classes -> ccot-axioms.xlsx


  to:   18 classes -> to-axioms.xlsx
